# small_fable — GROUNDING PROBE (frozen base model, inference-only)

**Question:** can a **FROZEN** base instruct model (`Qwen2.5-1.5B-Instruct` — *no training, no LoRA, no gradients*) **ground and follow** an abstract **multi-turn plan written in English**?

Plans are **English on purpose**: a custom primitive token (`READ`, `FILTER`, …) would carry an *untrained* embedding that means nothing to a frozen model. So the probe speaks the language the model already understands and measures whether it *uses* the plan. This separates two questions — *can the model ground an abstract plan?* (tested here, in English) vs *can we compress that into a learned primitive vocab?* (a later training question).

**Data:** 100 examples, 2–5 English turns, 8 varied topics (constraint-select, ordering, set-ops, transitive logic, scheduling, categorization, multi-hop lookup, conditional recommendation). Every answer is **known by construction** (a per-topic Python solver computes the gold answer *and* the wrong-plan's answer), so grading needs **no LLM judge**.

**The verdict (same 3-number test as the executor run):** for each problem we decode 3 ways — gold plan / no plan / wrong plan — and check:
- `acc_plan` — gold plan → gold answer
- **`neg_follow`** — wrong plan → *that wrong plan's* answer (the model faithfully followed it)
- **`neg_to_gold`** — wrong plan → gold answer (the model ignored the plan)
- `acc_noplan` — no plan → gold answer

**Grounding works iff `neg_follow` is HIGH and `neg_to_gold` is LOW** — but judged on the right rows. On problems the model can already solve **unaided**, a wrong plan is correctly *overridden* (the model just solves it), which looks like 'ignoring' but isn't a grounding failure. So the honest, headline number is the **CONDITIONAL** metric: `neg_follow` / `neg_to_gold` restricted to the rows where the model **fails without a plan** (`acc_noplan == 0`) — i.e. where the plan is actually load-bearing. The run prints **RAW** (all rows) and **CONDITIONAL** blocks; the **VERDICT is the conditional one**. Each prompt also tells the model to commit the asked-for entity (the item's *name*, not a price), so a correct decode isn't missed on a formatting slip.

## 0 · GPU check
Runs on CPU too (slower). On Colab: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi || echo 'NO GPU — runs on CPU (slow). Colab: Runtime -> Change runtime type -> T4 GPU, then re-run.'

## 1 · Clone & install
Same repo + `requirements.txt` the SSH user installs.

In [ ]:
import os
REPO = 'https://github.com/sinha-k-prat/small_fable.git'
if not os.path.isdir('small_fable'):
    !git clone -q $REPO
else:
    !cd small_fable && git pull -q
%cd small_fable
!pip install -q -r requirements.txt
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('\nReady.')

## 2 · Build the grounding dataset (deterministic, CPU, instant)
`tools/gen_grounding_data.py` builds 100 examples across 8 topics; each ships a gold plan, its answer, and a one-turn-perturbed wrong plan whose answer is also computed by construction (asserted different).

In [ ]:
!python -u tools/gen_grounding_data.py --n 100 --seed 7 --out grounding_test_data.jsonl
import json, collections
rows = [json.loads(l) for l in open('grounding_test_data.jsonl')]
print('rows:', len(rows))
print('topics:', dict(collections.Counter(r['topic'] for r in rows)))
print('turn-counts:', dict(collections.Counter(r['n_turns'] for r in rows)))
r = rows[0]
print('\nexample —', r['topic'], f"({r['n_turns']} turns)")
print(' problem:', r['problem'][:120])
print(' gold plan:', r['gold_plan'])
print(' gold:', r['gold_answer'], '| wrong-plan answer:', r['neg_answer'])

## 3 · Run the probe — `!python grounding_test.py` (the SAME file the SSH user runs)
The notebook does not re-implement anything; it shells out to the exact frozen-inference script. On **A100** set `DTYPE='bf16'`; on a **T4** keep `'fp16'` (T4 has no fast bf16).

In [ ]:
DTYPE = 'fp16'  # 'bf16' on A100
!python -u grounding_test.py \
    --data grounding_test_data.jsonl \
    --base Qwen/Qwen2.5-1.5B-Instruct \
    --dtype $DTYPE --device cuda \
    --max_new_tokens 320 \
    --out grounding_out

## 4 · The verdict plot
`grounding_out/grounding.png`: (A) overall, (B) per-topic, (C) per-turn-count. Watch `neg_follow` (want HIGH) vs `neg_to_gold` (want LOW).

In [ ]:
import os
from IPython.display import Image, display
png = 'grounding_out/grounding.png'
display(Image(filename=png)) if os.path.exists(png) else print(f'[!] {png} not found — did cell 3 finish?')

## 5 · `results.json` — metrics + grounding VERDICT

In [ ]:
import json, os
p = 'grounding_out/results.json'
if not os.path.exists(p):
    print('[!] results.json not found — re-run cell 3.')
else:
    res = json.load(open(p)); m = res['overall']; mr = res.get('marker_rate', 1.0)
    print(f'  marker_rate  {mr:.0%}' + ('' if mr>=0.9 else '  <-- LOW: raise --max_new_tokens; results unreliable'))
    print('base:', res['base'], '| rows:', res['n_rows'], '|', res['dtype'], res['device'])
    for k in ['acc_plan','neg_follow','neg_to_gold','acc_noplan']:
        print(f'  {k:12} {m[k]:.1%}')
    grounds = m['neg_follow'] >= 0.50 and m['neg_to_gold'] <= 0.30
    print(f"\nVERDICT: {res['verdict']}   [{res['verdict_rule']}]")
    print('  ->', 'GROUNDABLE: the frozen model follows English plans.' if grounds
          else 'NOT yet: the model ignores or cannot ground the plan.')

## 6 · Inspect a few decoded transcripts
For a few problems, the frozen model's committed answer under gold plan / no plan / wrong plan.

In [ ]:
import json, os
if os.path.exists('grounding_out/results.json'):
    for s in json.load(open('grounding_out/results.json')).get('examples', [])[:6]:
        print('TOPIC', s['topic'], '| gold', s['gold'], '| neg', s['neg_answer'])
        print('   plan  ->', repr(s['plan_decode'][-60:]), '  ok' if s['acc_plan'] else '  X')
        print('   neg   ->', repr(s['neg_decode'][-60:]), '  follow' if s['neg_follow'] else ('  to_gold' if s['neg_to_gold'] else '  other'))
        print('   noplan->', repr(s['noplan_decode'][-60:]))
        print()